# 27 RLOO baseline 如何降低 REINFORCE 方差？

## 面试回答主线

RLOO（REINFORCE Leave-One-Out）对同一 prompt 采样 K 条完整回复，对第 i 条用其余 K−1 条 reward 的均值作 baseline。这样 baseline 不含当前样本本身，能减少方差且不需要 critic。面试时要强调 baseline 应 stop-gradient，组必须来自同一 prompt，K=1 时 LOO 不成立。实验对两个客服 prompt 各采样三条回复，比较“包含自身的 group mean”与 leave-one-out advantage，并展示把不同 prompt 混组和 K=1 的失败。

**核心公式：** $A_i=r_i-\frac1{K-1}\sum_{j\ne i}r_j$；REINFORCE loss 可写为 $-A_i\sum_t\log\pi(y_{it}|x)$，优势和 baseline 不应反传梯度。

后续依次展示同数据基线、手写核心状态/概率、结果表、真实失败与修复。数值仅用于机制验证。


## 真实案例

数据是六条脱敏客服 prompt，每条含 chosen/rejected 回答；注意力主题会将它们映射成流式键值事件。字段语义和失败模式与真实系统一致，但样本规模不能代表线上效果。


In [1]:
import math  # 导入数学函数实现概率和复杂度公式。
import warnings  # 导入警告控制模块保持输出干净。
warnings.filterwarnings('ignore', message='The pynvml package is deprecated')  # 屏蔽环境依赖产生的弃用提示。
import torch  # 导入张量和自动微分基础能力。
import torch.nn as nn  # 导入模块基类以显式定义网络。
torch.manual_seed(41)  # 固定随机种子保证输出可复现。
torch.set_num_threads(1)  # 固定小实验 CPU 线程数。
samples = [  # 定义六条可读的 prompt、候选回复或流式事件。
    {'id': 'P01', 'prompt': '支付重复扣款怎么处理？', 'chosen': '核验订单后原路退款。', 'rejected': '无需核验直接忽略。'},  # 退款决策样本。
    {'id': 'P02', 'prompt': '发现陌生转账怎么办？', 'chosen': '立即冻结并核验身份。', 'rejected': '等待下个账单周期。'},  # 账户安全样本。
    {'id': 'P03', 'prompt': '收不到登录验证码？', 'chosen': '检查手机号并重发。', 'rejected': '建议注销账户。'},  # 登录支持样本。
    {'id': 'P04', 'prompt': '地址如何修改？', 'chosen': '在发货前更新地址。', 'rejected': '永久不可修改。'},  # 售后样本。
    {'id': 'P05', 'prompt': '银行卡被盗刷？', 'chosen': '冻结卡并保留证据。', 'rejected': '继续正常使用。'},  # 风险样本。
    {'id': 'P06', 'prompt': '发票抬头写错？', 'chosen': '按规则更正抬头。', 'rejected': '删除全部订单。'},  # 账单样本。
]  # 结束可读数据定义。
print('教学实验：六条脱敏客服 prompt/候选或流式状态，只解释训练与状态机制。')  # 声明实验边界。
for row in samples:  # 逐条展示 prompt/chosen/rejected。
    print(f"{row['id']} | 问题={row['prompt']} | chosen={row['chosen']} | rejected={row['rejected']}")  # 输出真实语义样本。


教学实验：六条脱敏客服 prompt/候选或流式状态，只解释训练与状态机制。
P01 | 问题=支付重复扣款怎么处理？ | chosen=核验订单后原路退款。 | rejected=无需核验直接忽略。
P02 | 问题=发现陌生转账怎么办？ | chosen=立即冻结并核验身份。 | rejected=等待下个账单周期。
P03 | 问题=收不到登录验证码？ | chosen=检查手机号并重发。 | rejected=建议注销账户。
P04 | 问题=地址如何修改？ | chosen=在发货前更新地址。 | rejected=永久不可修改。
P05 | 问题=银行卡被盗刷？ | chosen=冻结卡并保留证据。 | rejected=继续正常使用。
P06 | 问题=发票抬头写错？ | chosen=按规则更正抬头。 | rejected=删除全部订单。


## Baseline / 基线

先运行最朴素、但同样使用这些输入和同一指标的对照，避免只看一个核心算法数字。


In [2]:
groups = {'退款审批': [1.0, 0.3, 0.8], '盗刷处置': [1.0, 0.0, 0.7]}  # 定义每个 prompt 下三条完整回答的 reward。
mean_advantages = []  # 保存包含自身的 group-mean 优势。
for prompt, rewards in groups.items():  # 分别处理每个 prompt 组。
    mean_reward = sum(rewards) / len(rewards)  # 计算错误但常见的含自身均值。
    mean_advantages.extend([reward - mean_reward for reward in rewards])  # 构造 group mean advantage。
baseline_metric = sum(abs(value) for value in mean_advantages) / len(mean_advantages)  # 记录平均绝对优势幅度。
print(f'含自身 group-mean advantages={ [round(value, 3) for value in mean_advantages] }，平均绝对值={baseline_metric:.3f}')  # 展示基线信号。


含自身 group-mean advantages=[0.3, -0.4, 0.1, 0.433, -0.567, 0.133]，平均绝对值=0.322


## 手写核心实现与中间量

核心实现保留 state、ratio、优势、mask 或概率分母等中间量，不用 Trainer 或现成 Agent/Attention 框架遮蔽机制。


In [3]:
loo_advantages = []  # 保存 leave-one-out 优势。
for prompt, rewards in groups.items():  # 逐 prompt 构造独立 LOO baseline。
    for index, reward in enumerate(rewards):  # 逐条候选计算它自己的 baseline。
        others = [value for other_index, value in enumerate(rewards) if other_index != index]  # 排除当前样本 reward。
        loo_baseline = sum(others) / len(others)  # 计算其余候选的平均 reward。
        loo_advantages.append(reward - loo_baseline)  # 得到不含自身的 RLOO advantage。
core_metric = sum(abs(value) for value in loo_advantages) / len(loo_advantages)  # 保存 LOO 平均绝对优势。
print(f'RLOO advantages={ [round(value, 3) for value in loo_advantages] }，平均绝对值={core_metric:.3f}')  # 展示逐样本 LOO 差异。


RLOO advantages=[0.45, -0.6, 0.15, 0.65, -0.85, 0.2]，平均绝对值=0.483


In [4]:
comparison_rows = [('Baseline', float(baseline_metric)), ('核心机制', float(core_metric))]  # 建立基线与核心的同口径结果表。
for name, metric in comparison_rows:  # 逐行输出结果表。
    print(f'{name:<8} | 指标={metric:.6f}')  # 显示可读数值对照。


Baseline | 指标=0.322222
核心机制     | 指标=0.483333


## 结果解读

这里只能得出本受控样本上的机制结论。生产要保存 prompt group id、rollout 版本和 reward 版本；同一 prompt 的候选数不齐时需要明确处理策略，而非静默填零。 生产决策必须进一步看验证集、线上安全指标、算力和版本可追溯性。

## 失败案例

下方先让关键条件真实失效，再展示修复如何改变可观测指标。


In [5]:
singleton_rewards = [0.6]  # 构造只有一条 rollout 的 prompt 组。
failure_metric = int(len(singleton_rewards) - 1 == 0)  # 标记 K=1 时 LOO 分母为零的失败条件。
fallback_baseline = 0.5  # 显式选择离线全局基线作为降级策略。
fix_metric = singleton_rewards[0] - fallback_baseline  # 计算降级后仍可定义的优势。
print(f'失败：K=1 时没有其他样本可做 LOO={bool(failure_metric)}；修复：使用显式 fallback advantage={fix_metric:.3f}')  # 展示不能隐式除零。


失败：K=1 时没有其他样本可做 LOO=True；修复：使用显式 fallback advantage=0.100


## 工程取舍、常见坑与延伸追问

**工程取舍：** 生产要保存 prompt group id、rollout 版本和 reward 版本；同一 prompt 的候选数不齐时需要明确处理策略，而非静默填零。

**常见坑：** 用含自身的 group mean 当 LOO，或把不同 prompt、不同 reward rubric 的样本混在同一 baseline 里。

**延伸追问：** RLOO 与 critic baseline 的方差/成本如何比较？K 增大为何既能稳优势又会增加 rollout 成本？

## 生产差距

实验运行于 CPU/FP32，只有 6 条离线样本，省略了真实 rollout、分布式同步、混合精度、内容安全、数据治理、checkpoint 和监控。上线版本应以受审计的状态、指标和回滚流程替代这些教学变量。


In [6]:
assert len(loo_advantages) == 6  # 验证六条 rollout 都有 LOO 优势。
assert failure_metric == 1  # 验证 K=1 是真实的 LOO 边界条件。
assert abs(fix_metric) > 0.0  # 验证 fallback 产生了定义明确的优势。
assert core_metric > 0.0  # 验证组内 reward 差异提供了训练信号。
